### Code funzionante

In [530]:
import math
import numpy as np
import matplotlib.pyplot as plt
from random import randrange
import random

In [531]:
bits = 128

In [535]:
a = 142265943202673294119456155468653722800
b = 32123312

LUT_BITS = 16
LUT_SIZE = 1 << LUT_BITS

lut = [
    (1 << (bits * 2)) //
    ((1 << (bits - 1)) + (i << (bits - 1 - LUT_BITS)))
    for i in range(LUT_SIZE)
]

We now need to compute `b.bit_length()` homomorphically

In [554]:
"""
Requires log(bits) multiplications to simulate the XOR
Q: Is there an algorithm that can "mask" the first 1 and mask out all the rest?
"""
def homomorphic_bit_length(arr):
    n = len(arr)
    padded = np.concatenate([arr, np.zeros(n, dtype=int)])  # [arr | 000...0]
    
    step = 1
    while step < n:
        padded = np.add(padded, np.roll(padded, step))
        step *= 2
        
    print(padded[:n])
    
    return int(padded[:n].sum())  # prendi solo la prima metà

In [555]:
np.array(list(np.binary_repr(b, width=bits)), dtype=int)

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0,
       1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0])

In [556]:
bit_length_fhe = homomorphic_bit_length(np.array(list(np.binary_repr(b, width=bits)), dtype=int))

s = bits - bit_length_fhe

print("s:", s)

[ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0
  0  0  0  0  0  0  0  1  2  3  4  4  5  5  6  6  6  6  7  7  8  8  8  9
 10 10 11 12 12 12 12 12]
s: -58


In [538]:
2**7

128

Given the encrypted `b._bit_length()`, we now need to perform a `blind shift` as:

```
b_norm = b << s
```


Ideas:
1) Converto in binario s (occupa 7 bits a 128-bits), ruoto numero logaritmico e maschero SI/NO se il bit corrente di s è 1

In [501]:
s_bin = np.array(list(np.binary_repr(s, width=8)), dtype=int) # In FHE si fa con polinomi, magari a 119 degree è ok visto che siamo in [0, 128]

s_bin = s_bin[::-1]
s_bin

array([0, 0, 0, 0, 0, 0, 0, 0])

A questo punto credo che abbiamo fatto (in 128 bits) 14 moltiplicazioni... quindi si bootstrappa

In [502]:
"""
Requires log(bits) multiplications to mask
"""

b_norm_rot = np.array(list(np.binary_repr(b, width=128)), dtype=int)[::-1]

for i in range(7):
    mask = np.ones(128)
    mask *= s_bin[i]
    
    b_norm_rot = b_norm_rot * (1 - mask) + np.roll(b_norm_rot, 2**i) * mask
    
b_norm_rot

array([0., 1., 1., 0., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 0., 0.,
       0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0.,
       1., 1., 0., 1., 1., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 1., 1.,
       1., 1., 0., 1., 0., 1., 1., 1., 0., 1., 1., 1., 0., 0., 0., 0., 0.,
       1., 1., 0., 1., 1., 0., 0., 1., 0., 1., 1., 1., 0., 0., 0., 1., 1.,
       0., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 1., 0., 1., 0., 0.,
       1., 0., 1., 1., 0., 0., 0., 0., 1., 0., 1., 0., 1., 1., 1., 1., 1.,
       0., 1., 0., 0., 0., 1., 0., 1., 1.])

In [503]:
b_norm = int(''.join(b_norm_rot[:bits][::-1].astype(int).astype(str)), 2)
b_norm

278459053012191913190882327792566697846

In [504]:
b_norm_rot = np.concatenate([b_norm_rot, np.zeros(bits, dtype=int)])  # [arr | 000...0]
b_norm_rot = np.roll(b_norm_rot, -(bits - 1 - LUT_BITS))

In [505]:
mask = np.zeros(len(b_norm_rot))
mask[:LUT_BITS] = 1
b_norm_rot = b_norm_rot * mask
idx = int(''.join(b_norm_rot[:bits][::-1].astype(int).astype(str)), 2)
print("idx", idx)

idx 41722


In [506]:
b_norm_rot

array([0., 1., 0., 1., 1., 1., 1., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
       0., 0., 0., 0., 0.

In [507]:
x = lut[idx]
print("x:", x)

x: 415833694428958644418798006165476915586


In [508]:
for _ in range(3):
    x = (x * ((1 << (bits * 2 + 1)) - b_norm * x)) >> (bits * 2)

result = (a * x) >> ((bits * 2) - s)

print("A:", a)
print("B:", b)
print("My res:", result)
print("Expect:", a // b)

A: 142265943202673294119456155468653722800
B: 278459053012191913190882327792566697846
My res: 0
Expect: 0


### Todo: map the LUT as a polynomial

In [465]:
def get_bit(n, i):
    return (n >> i) & 1

In [466]:
def polylut(bit = 0):
    N = LUT_SIZE
    seq = np.array([get_bit(lut[i], bit) for i in range(N)], dtype=float)
    c = np.fft.fft(seq)  

    def f(x):
        s = 0.0
        for k in range(N):
            ang = 2 * math.pi * k * x / N
            s += c[k].real * math.cos(ang) - c[k].imag * math.sin(ang)
        return s / N

    return f

In [467]:
class Chebyshev:
    """
    Chebyshev(a, b, n, func)
    Given a function func, lower and upper limits of the interval [a,b],
    and maximum degree n, this class computes a Chebyshev approximation
    of the function.
    Method eval(x) yields the approximated function value.
    """
    def __init__(self, a, b, n, func):
        n = n + 1
        self.a = a
        self.b = b
        self.func = func

        bma = 0.5 * (b - a)
        bpa = 0.5 * (b + a)
        f = [func(math.cos(math.pi * (k + 0.5) / n) * bma + bpa) for k in range(n)]
        self.roots = f

        self.x = [math.cos(math.pi * (k + 0.5) / n) * bma + bpa for k in range(n)]

        fac = 2.0 / n
        self.c = [fac * sum([f[k] * math.cos(math.pi * j * (k + 0.5) / n)
                  for k in range(n)]) for j in range(n)]

    def eval(self, x):
        a,b = self.a, self.b
        #assert(a <= x <= b)
        y = (2.0 * x - a - b) * (1.0 / (b - a))
        y2 = 2.0 * y
        (d, dd) = (self.c[-1], 0)             # Special case first step for efficiency
        for cj in self.c[-2:0:-1]:            # Clenshaw's recurrence
            (d, dd) = (y2 * d - dd + cj, d)
        return y * d - dd + 0.5 * self.c[0]   # Last step is different

In [468]:
# SKIPPALO SE GIA AVVIATO, PESANTE 

polyluts = []

for bit in range(bits + 2):
    f = polylut(bit)
    
    c = Chebyshev(0, LUT_SIZE, 851, f)
    
    polyluts.append(c)

KeyboardInterrupt: 

In [ ]:
f = polylut(0)

c = Chebyshev(0, LUT_SIZE, 851, f)

x = np.arange(0, LUT_SIZE, 1)

y = np.vectorize(c.eval)(x)
plt.plot(x, y)

y_real = np.vectorize(f)(x)
plt.plot(x, y_real)


max(abs(y - y_real))

---

### Making the LUT polynomial

In [ ]:
binx = []

for i in range(bits * 2):
    if i >= len(polyluts):
        binx.append(0)
    else:
        binx.append(polyluts[i].eval(idx))

In [ ]:
polydec = ([round(x) for x in list(binx)][::-1])

In [ ]:
n = int("".join(map(str, polydec)), 2)

# We check that the polynomial LUT and the actual LUT are equal
n == lut[idx]

In [ ]:
x = n

for _ in range(3):
    term = b_norm * x                  # Mult
    term = 2 ** (bits * 2 + 1) - term  # Implementato come !term + 1, dove i bit più alti di 33 sono a 0
    
    x = x * term                       # Mult normale
    x = x >> bits * 2                  # Shift gratis

result = (a * x) >> (bits * 2 - s)

print(result)
print(a // b)

In [ ]:
##
## Expected: 3.5 minutes to evaluate (optimization might be smaller bootstrap tables)
##